# Programação Linear



In [75]:
using JuMP, Cbc, Clp, Juniper, Ipopt

# Maximização do Lucro

Num livro de receitas existem diferentes produtos que precisam dos seguintes ingredientes , tempo de preparo, porções, preço

- Geleia de Limao: 2 peptina, 3 limao , 3h, 5p, 10R$
- Geleia de morango: 1 peptina, 4 morangos, 2h, 3p, 12R$
- Mousse de limao: 4 limoes, 4 morangos, 4h, 10p, 5$

os produtos custam:

- pepitna, 1R$
- limao, 2R$
- morango, 4R$

Tempo disponivel para trabalho: 2 semanas (80h), 100R$



In [153]:
n_semana = 2
dias_por_semana = 4
capacidade_de_investimento = 1000

1000

In [156]:
model = Model(Cbc.Optimizer) #Branch and Bound
        #A              #B              #C
P = ["geleia_limao","geleia_morango","mousse_limao"]
M = ["peptina","limao","morango"]

Tmax = n_semana*dias_por_semana*10

#Entradas
pedidos_confirmados = [10;8;2]
preco_de_venda = [15;12;5]

# Saidas
quantidade_inicial_de_material = [0;0;0]

custo_de_ingrediente = [5;8;4]

                    #Produto A B C  
quantidade_de_ingrediente = [1 2 0; #Peptina
                             4 0 6; #Limao
                             0 8 4] #Morango

tempo_de_producao = [6;8;5]
porcoes_por_producao = [20;15;2]
custo_sobra = (0.3)*preco_de_venda

NP = length(P)
NM = length(M)

# Variaveis
@variable(model, total_produtos_vendidos >= 0, Int)
@variable(model, produtos_vendidos[1:NP] >= 0, Int) #Quantidade de produto p pertencente a cada P vendido
@variable(model, ingredientes_comprados[1:NM] >= 0, Int) #Quantidade de ingredientes i pertencente a cada I comprado
@variable(model, n_producao_realizada[1:NP] >= 0, Int)
@variable(model, qtde_disponivel[1:NP] >= 0, Int)
@variable(model, sobra[1:NP] >= 0, Int)
#FOB

@objective(model, Max, 
                        sum(produtos_vendidos[p] * preco_de_venda[p] for p = 1:NP)
                        - sum(ingredientes_comprados[i] * custo_de_ingrediente[i] for i= 1:NM)
                        - sum(sobra[p] * custo_sobra[p] for p=1:NP)
                        )


                    
# Restricoes
@constraint(model, calculo_total, 
            total_produtos_vendidos == sum(produtos_vendidos[p] for p=1:NP)               
          )

@constraint(model, produtos_disponiveis[p=1:NP], 
              qtde_disponivel[p] == n_producao_realizada[p]*porcoes_por_producao[p]               
          )

@constraint(model, Balanco[p=1:NP], 
            qtde_disponivel[p] - produtos_vendidos[p] ==  sobra[p]               
        )

@constraint(model, Atendimento[p=1:NP], 
     produtos_vendidos[p] >= pedidos_confirmados[p] 
    )

@constraint(model, lim_tempo, 
                sum(n_producao_realizada[p] * tempo_de_producao[p] for p=1:NP)
                <=
                Tmax
            )

@constraint(model, lim_material[i=1:NM],
                  sum(quantidade_de_ingrediente[i,p] * n_producao_realizada[p] for p=1:NP) 
                  == 
                  quantidade_inicial_de_material[i] + ingredientes_comprados[i]         
              )
    

@constraint(model, limite_investimento,
            sum(ingredientes_comprados[i] * custo_de_ingrediente[i] for i=1:NM) 
            <= 
            capacidade_de_investimento
           )

@constraint(model, variedade_produto[p=1:NP],
           produtos_vendidos[p]
           <= 
           (0.4)*total_produtos_vendidos
          )

#Otimizacao
print(model)
status = optimize!(model)

#Pos Otimizacao
produtos_vendidos_result = value.(produtos_vendidos)
ingredientes_comprados_result = value.(ingredientes_comprados)
sobra_result = value.(sobra)
n_producoes_feitas = value.(n_producao_realizada)
qtde_disponivel_result = value.(qtde_disponivel)
invest_result = sum(value.(ingredientes_comprados[i]) * custo_de_ingrediente[i] for i=1:NM) 
tempo_gasto_result = sum(value.(n_producao_realizada[p]) * tempo_de_producao[p] for p=1:NP)

fob_result = objective_value(model)
fob_status = termination_status(model)


Max 15 produtos_vendidos[1] + 12 produtos_vendidos[2] + 5 produtos_vendidos[3] - 5 ingredientes_comprados[1] - 8 ingredientes_comprados[2] - 4 ingredientes_comprados[3] - 4.5 sobra[1] - 3.5999999999999996 sobra[2] - 1.5 sobra[3]
Subject to
 calculo_total : total_produtos_vendidos - produtos_vendidos[1] - produtos_vendidos[2] - produtos_vendidos[3] = 0
 produtos_disponiveis[1] : -20 n_producao_realizada[1] + qtde_disponivel[1] = 0
 produtos_disponiveis[2] : -15 n_producao_realizada[2] + qtde_disponivel[2] = 0
 produtos_disponiveis[3] : -2 n_producao_realizada[3] + qtde_disponivel[3] = 0
 Balanco[1] : -produtos_vendidos[1] + qtde_disponivel[1] - sobra[1] = 0
 Balanco[2] : -produtos_vendidos[2] + qtde_disponivel[2] - sobra[2] = 0
 Balanco[3] : -produtos_vendidos[3] + qtde_disponivel[3] - sobra[3] = 0
 lim_material[1] : -ingredientes_comprados[1] + n_producao_realizada[1] + 2 n_producao_realizada[2] = 0
 lim_material[2] : -ingredientes_comprados[2] + 4 n_producao_realizada[1] + 6 n_produca

OPTIMAL::TerminationStatusCode = 1

In [157]:

println(repeat("-",70))
for p = 1:NP
    println(" $(P[p]) vendidos = $(produtos_vendidos_result[p])")
end

println(repeat("-",70))
for p = 1:NP
    println(" Dos produtos $(P[p])  = $(qtde_disponivel_result[p]) disponivel")
end

println(repeat("-",70))
for p = 1:NP
    println(" Foram Produção de $(P[p])  = $(n_producoes_feitas[p])")
end

println(repeat("-",70))
for i = 1:NM
    println(" $(M[i]) comprados = $(ingredientes_comprados_result[i])")
end

println(repeat("-",70))
for p = 1:NP
    println(" $(P[p]) que sobrou = $(sobra_result[p])")
end


println(repeat("-",70))
println()
println(" Lucro Total = $fob_result R\$ [$(100*(fob_result/invest_result))] %")
println(" FOB_status = $fob_status")
println(" Tempo Trabalhado Total = $tempo_gasto_result h\$")
println(" Dinheiro Investido = $invest_result R\$")
println()
println(repeat("-",70))
println()

----------------------------------------------------------------------
 geleia_limao vendidos = 30.0
 geleia_morango vendidos = 30.0
 mousse_limao vendidos = 16.0
----------------------------------------------------------------------
 Dos produtos geleia_limao  = 40.0 disponivel
 Dos produtos geleia_morango  = 30.000000000000004 disponivel
 Dos produtos mousse_limao  = 16.0 disponivel
----------------------------------------------------------------------
 Foram Produção de geleia_limao  = 2.0
 Foram Produção de geleia_morango  = 2.0
 Foram Produção de mousse_limao  = 8.0
----------------------------------------------------------------------
 peptina comprados = 6.0
 limao comprados = 56.0
 morango comprados = 47.99999999999999
----------------------------------------------------------------------
 geleia_limao que sobrou = 10.0
 geleia_morango que sobrou = 0.0
 mousse_limao que sobrou = 0.0
----------------------------------------------------------------------

 Lucro Total = 175.0 R$ 